# qbraid.runtime.quobly

> **Billing notice:** the Quobly Alloy Forge emulator is currently **free to run** (0 credits). You need a qBraid API key, but no Quobly account.

[Quobly](https://quobly.io) builds quantum processors from **electron spins in silicon**, manufactured on standard CMOS processes — the idea being that the semiconductor industry's existing infrastructure is what makes a fault-tolerant machine scalable.

**Alloy Forge** is their physics-based emulator of **Pioneer**, Quobly's 10-qubit silicon spin-qubit QPU. It reproduces the hardware's actual error behaviour, and it runs wider than the hardware does: up to **15 qubits**, on a **linear nearest-neighbour** coupling map.

This notebook submits a Bell pair twice — once with the noise model off, once with it on — and shows what is different about reading results back from this device.

In [ ]:
%%capture

# "%%capture" hides the install output; comment it out if you need to debug this step.
%pip install 'qbraid[visualization]' qiskit

## Connect to the device

`QbraidProvider` picks up your API key from `~/.qbraid/qbraidrc` or the `QBRAID_API_KEY`
environment variable. Inside qBraid Lab it is already configured for you.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
device = provider.get_device("qbraid:quobly:sim:alloy-forge")

device.metadata()

15 qubits, `ONLINE`, and `perShot` / `perTask` pricing of zero.

One limit `metadata()` does not surface: **1000 shots** is the ceiling on this device. `qbraid devices get qbraid:quobly:sim:alloy-forge` in a terminal shows it, along with `minShots` and the queue depth.

In [ ]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)      # qubits 0 and 1 are adjacent on the linear array
circuit.measure_all()

circuit.draw()

## Run it without noise

Device options for Alloy Forge go in a **`runtime_options` dict**, not in the `run()` signature.

```python
device.run(circuit, shots=200, noise=False)   # TypeError
device.run(circuit, shots=200, runtime_options={"noise": False})   # correct
```

There are three: `noise` (bool), `seed` (int), and `shots` — which is the exception, passed
directly to `run()`.

A `seed` makes the run exactly reproducible, noise included, so the numbers below are the same
every time you execute this notebook.

In [ ]:
ideal_job = device.run(circuit, shots=200, runtime_options={"noise": False, "seed": 1234})
ideal_counts = ideal_job.result().data.get_counts()

ideal_counts

### Reading the counts

Results come back at **your circuit's own width** — a two-qubit circuit gives two-bit keys. Qubit
ordering is little-endian: **qubit 0 is the rightmost bit**.

So the two peaks above are `|00⟩` and `|11⟩`, exactly as a Bell state should be.

## Run it *with* the Pioneer noise model

Here is the one behaviour that catches everybody: **`noise` defaults to `True`**. A plain
`device.run(circuit, shots=200)` is already a noisy run. The noiseless run above needed the
explicit opt-out.

Same circuit, same seed, noise on:

In [ ]:
noisy_job = device.run(circuit, shots=200, runtime_options={"noise": True, "seed": 1234})
noisy_counts = noisy_job.result().data.get_counts()

noisy_counts

Almost identical to the noiseless run — and that is the honest result. A Bell pair is one
two-qubit interaction deep, and Pioneer's error rate at that depth is well under 1%, so at 200
shots the error outcomes often do not appear at all.

That is worth knowing before you go looking for noise: **a shallow circuit on this emulator looks
clean because the hardware it models is good at shallow circuits.** To see the noise model work,
you have to give it something to accumulate over.

In [ ]:
# A GHZ chain: one extra two-qubit interaction per qubit added.
# Every gate is between neighbours, so nothing needs routing.
def ghz(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure_all()
    return qc


deep_job = device.run(ghz(6), shots=200, runtime_options={"noise": True, "seed": 1234})
deep_counts = deep_job.result().data.get_counts()

valid = deep_counts.get("000000", 0) + deep_counts.get("111111", 0)
print(f"shots on a valid GHZ outcome: {valid / 200:.1%}")
print(f"distinct outcomes observed  : {len(deep_counts)}")

Six qubits deep, only about **78%** of shots still land on `|000000⟩` or `|111111⟩`. The rest are
states a perfect GHZ can never produce — that population is the silicon spin-qubit error model
doing its job.

Plotted against the ideal two-peak distribution:

In [ ]:
from qbraid.visualization import plot_histogram

ideal_deep = device.run(ghz(6), shots=200, runtime_options={"noise": False, "seed": 1234})

plot_histogram(
    [ideal_deep.result().data.get_counts(), deep_counts],
    legend=["noise=False", "Pioneer noise model"],
    title="6-qubit GHZ chain on Quobly Alloy Forge",
)

## Two things to know before you write your own circuits

**Keep two-qubit gates between adjacent qubits.** Pioneer's `RZZ` interaction only exists between
neighbours on the linear array — 0–1, 1–2, 2–3, and so on. A gate spanning non-adjacent qubits is
accepted and runs, but the counts come back measured on different qubits than the ones you named,
so the result will not line up with your indices. Build along the chain, or insert your own `SWAP`s.

What actually decides this is the transpiler. To satisfy the connectivity constraint it may place
your logical qubits onto different physical ones, and the emulator reports positions on the
physical register rather than yours. A chain keeps that placement one-to-one, which is why every
circuit in these notebooks is written as a chain.

**Shot count drives the wall clock.** The emulator integrates the physics shot by shot, so a job's
runtime tracks `shots` more closely than it tracks circuit width — the same 4-qubit circuit takes
about 30 s at 100 shots and about 135 s at 1000. Batch submission is not supported, so a sweep runs
sequentially. Develop at 100–200 shots and raise it for the final run.

## Where next

- [Quobly Alloy Forge documentation](https://docs.qbraid.com/v2/sdk/user-guide/providers/native/quobly)
- `quobly_ghz_noise_benchmark.ipynb`, alongside this one — grows a GHZ chain along
  the coupling map and measures how fast the noise accumulates.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>